# Biomarker selection for downstream validation

This notebook is the endpoint of the extracted workflow. The original
analysis lives in `src/prioritization/original.py` and the copied
modules under `src/modules/`. Run the full computation with:

```bash
python scripts/run_pipeline.py --config config/analysis_config.yaml
```

The cell below only checks the three ranking matrices when those files
are already at the configured paths.


In [ ]:
from pathlib import Path
import sys
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from prioritization.original import FEATURE_RANKERS, AGGREGATORS
from prioritization.matrices import load_ranking_datasets

print(len(FEATURE_RANKERS), "rankers")
print(len(AGGREGATORS), "ensemble aggregators")
cfg = yaml.safe_load((ROOT / "config" / "analysis_config.yaml").read_text())
paths = {k: str(ROOT / v) for k, v in cfg["processed"].items()}
missing = [p for p in paths.values() if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(
        "Processed matrices are not in data/processed. See data/raw/README.md.\n"
        + "\n".join(missing)
    )
data = load_ranking_datasets(paths)
for name, frame in data.items():
    print(name, frame.shape, frame["Diagnosis"].value_counts().to_dict())
